# TrustLens — Machine Learning & Risk Intelligence
### PS1: AI-Driven Personal Lending Risk & Trust Profiling

This notebook covers:
1. **Exploratory Data Analysis (EDA)** on borrower credit risk records.
2. **Data Cleaning & Preprocessing Pipeline** (Missing value imputation, categorical one-hot encoding, feature scaling).
3. **Model Training & Evaluation** (Random Forest Risk Classifier with class balance).
4. **Feature Importance Analysis** (Top drivers influencing credit risk).
5. **Trust Score & Explainability Verification** (Testing canonical PRD borrower scenarios).

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve
)

from train import load_and_clean_data, build_preprocessing_pipeline
from trust_engine import TrustEngine
from predict import TrustLensPredictor

print("Libraries successfully imported!")

## 1. Load & Inspect Dataset

In [ ]:
df = pd.read_csv("credit_risk_dataset.csv")
print(f"Dataset shape: {df.shape}")
df.head()

### Target Distribution (Loan Status: 0 = Non-Default, 1 = Default)

In [ ]:
print(df['loan_status'].value_counts(normalize=True))
df.isnull().sum()

## 2. Preprocess & Split Data

In [ ]:
df_clean = load_and_clean_data("credit_risk_dataset.csv")
X = df_clean.drop(columns=['loan_status'])
y = df_clean['loan_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Cleaned training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

## 3. Load Trained Model & Evaluate Performance

In [ ]:
pipeline = joblib.load("artifacts/risk_model_pipeline.joblib")
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("=" * 40)
print("Model Evaluation on Held-Out Test Set")
print("=" * 40)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Default', 'Default']))

## 4. Feature Importance Breakdown

In [ ]:
with open("artifacts/model_metadata.json", "r") as f:
    metadata = json.load(f)

top_feats = pd.DataFrame(metadata["top_features"])
print(top_feats)

## 5. End-to-End Decision Support Testing (PRD Scenarios)

In [ ]:
predictor = TrustLensPredictor()

# Borrower A: Low Risk Profile
borrower_a = {
    "income": 600000,
    "employment_years": 5,
    "loan_amount": 80000,
    "loan_purpose": "MEDICAL",
    "credit_history_years": 5,
    "previous_default": False
}

res_a = predictor.predict(borrower_a)
print("Borrower A Prediction:")
print(json.dumps(res_a, indent=2))